# Exploración de Qbeast sobre Delta Lake

Este notebook es independiente del benchmark de `Iceberg/dremio-file-ingestion-practice`: Qbeast no se integra con catálogos Iceberg/Nessie, requiere Delta Lake y su propio `QbeastCatalog` como `spark_catalog`. Aquí escribimos la misma tabla como Delta "plano" y como Qbeast (indexado), y comparamos cuántos ficheros lee cada uno ante una consulta filtrada.

**Dataset:** NYC Yellow Taxi Trip Records (Enero 2024).

## 1. Sesión de Spark

La configuración (paquetes de Delta y Qbeast, extensiones, catálogo) se carga desde `spark-defaults.conf`, copiado en la imagen.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
import time

spark = (
    SparkSession.builder
    .appName("Qbeast-Delta-Exploracion")
    .getOrCreate()
)

print(f"Spark session creada. Versión: {spark.version}")

Spark session creada. Versión: 3.5.0


## 2. Descarga y preparación del dataset

Spark no puede leer un `https://` directamente (falla con `UnsupportedOperationException: hasn't implemented listStatus`), así que descargamos el fichero a disco local del contenedor primero.

In [2]:
import os
import requests

DATASET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
LOCAL_PATH = "/home/jovyan/work/data/yellow_tripdata_2024-01.parquet"

os.makedirs(os.path.dirname(LOCAL_PATH), exist_ok=True)

if not os.path.exists(LOCAL_PATH):
    print(f"Descargando {DATASET_URL} ...")
    resp = requests.get(DATASET_URL, timeout=120)
    resp.raise_for_status()
    with open(LOCAL_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Descargado en {LOCAL_PATH} ({os.path.getsize(LOCAL_PATH):,} bytes)")
else:
    print(f"Ya existe {LOCAL_PATH}, se omite descarga.")

Descargando https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet ...
Descargado en /home/jovyan/work/data/yellow_tripdata_2024-01.parquet (49,961,641 bytes)


In [3]:
raw_df = spark.read.parquet(f"file://{LOCAL_PATH}")

df = (
    raw_df.select(
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        col("passenger_count").cast("int"),
        col("trip_distance").cast("double"),
        col("PULocationID").cast("int"),
        col("DOLocationID").cast("int"),
        col("fare_amount").cast("double"),
        col("tip_amount").cast("double"),
        col("total_amount").cast("double"),
    )
    .withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))
    .filter(col("fare_amount") > 0)
    .dropna()
)

df.cache()
print(f"Registros preparados: {df.count():,}")
df.printSchema()

Registros preparados: 2,788,237
root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_date: date (nullable = true)



## 3. Escritura: Delta plano (referencia) vs. Qbeast (indexado)

In [5]:
DELTA_PATH = "/home/jovyan/warehouse/delta/nyc_taxi_delta"
QBEAST_PATH = "/home/jovyan/warehouse/qbeast/nyc_taxi_qbeast"

start = time.time()
(df.write
 .format("delta")
 .mode("overwrite")
 .save(DELTA_PATH))
print(f"Delta plano escrito en {time.time() - start:.2f}s")

Delta plano escrito en 1.74s


In [7]:
columns_to_index = "PULocationID,DOLocationID,fare_amount"

start = time.time()
(df.write
 .format("qbeast")
 .mode("overwrite")
 .option("columnsToIndex", columns_to_index)
 .option("cubeSize", 50000)
 .save(QBEAST_PATH))
print(f"Qbeast escrito en {time.time() - start:.2f}s (columnas indexadas: {columns_to_index})")

Qbeast escrito en 6.00s (columnas indexadas: PULocationID,DOLocationID,fare_amount)


## 4. Comparativa: ficheros leídos ante una consulta filtrada

El valor real de Qbeast es reducir los datos leídos (*data skipping*) en consultas selectivas sobre las columnas indexadas, no necesariamente la velocidad de escritura.

In [8]:
FILTER_CONDITION = "PULocationID = 161 AND fare_amount > 20"

delta_filtered = spark.read.format("delta").load(DELTA_PATH).where(FILTER_CONDITION)
qbeast_filtered = spark.read.format("qbeast").load(QBEAST_PATH).where(FILTER_CONDITION)

delta_count = delta_filtered.count()
delta_files = len(delta_filtered.inputFiles())

qbeast_count = qbeast_filtered.count()
qbeast_files = len(qbeast_filtered.inputFiles())

print(f"Filtro: {FILTER_CONDITION}\n")
print(f"Delta plano -> filas: {delta_count:,} | ficheros leídos: {delta_files}")
print(f"Qbeast      -> filas: {qbeast_count:,} | ficheros leídos: {qbeast_files}")

Filtro: PULocationID = 161 AND fare_amount > 20

Delta plano -> filas: 21,381 | ficheros leídos: 3
Qbeast      -> filas: 21,381 | ficheros leídos: 27


In [9]:
print("--- Plan físico Delta plano ---")
delta_filtered.explain("formatted")

print("\n--- Plan físico Qbeast ---")
qbeast_filtered.explain("formatted")

--- Plan físico Delta plano ---
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [10]: [tpep_pickup_datetime#8875, tpep_dropoff_datetime#8876, passenger_count#8877, trip_distance#8878, PULocationID#8879, DOLocationID#8880, fare_amount#8881, tip_amount#8882, total_amount#8883, pickup_date#8884]
Batched: true
Location: PreparedDeltaFileIndex [file:/home/jovyan/warehouse/delta/nyc_taxi_delta]
PushedFilters: [IsNotNull(PULocationID), IsNotNull(fare_amount), EqualTo(PULocationID,161), GreaterThan(fare_amount,20.0)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:int,trip_distance:double,PULocationID:int,DOLocationID:int,fare_amount:double,tip_amount:double,total_amount:double,pickup_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [10]: [tpep_pickup_datetime#8875, tpep_dropoff_datetime#8876, passenger_count#8877, trip_distance#8878, PULocationID#8879, DOLocationID#8880, fa

## 5. Inspección del índice de Qbeast

La API `QbeastTable` es Scala; se accede vía el puente `py4j` de PySpark (`spark._jvm`).

In [10]:
qbeast_table = spark._jvm.io.qbeast.spark.QbeastTable.forPath(spark._jsparkSession, QBEAST_PATH)

print("Columnas indexadas:", qbeast_table.indexedColumns())
print("Revisión más reciente:", qbeast_table.latestRevisionID())
print("\nMétricas del índice:")
print(qbeast_table.getIndexMetrics())

Columnas indexadas: List(PULocationID, DOLocationID, fare_amount)
Revisión más reciente: 1

Métricas del índice:
OTree Index Metrics:
revisionId: 1
elementCount: 2788237
dimensionCount: 3
desiredCubeSize: 50000
indexingColumns: PULocationID:linear,DOLocationID:linear,fare_amount:linear
height: 6 (3)
avgFanout: 4.62 (8.0)
cubeCount: 126
blockCount: 126
fileCount: 27
bytes: 56520045

Multi-block files stats:
cubeElementCountStats: (count: 126, avg: 22128, stddev: 20602, quartiles: [1,1182,15830,45867,58510])
blockElementCountStats: (count: 126, avg: 22128, stddev: 20602, quartiles: [1,1182,15830,45867,58510])
fileBytesStats: (count: 27, avg: 2093335, stddev: 931110, quartiles: [1000758,1171840,1979908,3045365,4147411])
blockCountPerCubeStats: (count: 126, avg: 1, stddev: 0, quartiles: [1,1,1,1,1])
blockCountPerFileStats: (count: 27, avg: 4, stddev: 1, quartiles: [1,4,5,5,7])

Inner cubes depth-wise stats:
+----------------------------------------+----------------------------------------+

## 6. Limpieza

In [11]:
df.unpersist()
spark.stop()
print("Sesión de Spark detenida.")

Sesión de Spark detenida.
